# Comparación de Modelos (2018–2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.

El objetivo es comparar el desempeño de los modelos candidatos desarrollados durante la etapa de modelado (**Random Forest, XGBoost y Multilayer Perceptron (MLP)**) para seleccionar la alternativa con mejor capacidad predictiva en la clasificación de la severidad de hechos de tránsito. Esta etapa permite evaluar métricas de rendimiento, analizar fortalezas y limitaciones de cada enfoque, y definir el modelo final del sistema predictivo.


In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (roc_curve, roc_auc_score,
                             accuracy_score, f1_score, 
                             precision_score, recall_score)
from sklearn.utils.class_weight import compute_sample_weight
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de modelos y datos

In [2]:
# -- Cargar modelos entrenados ----------------------------------------------
rf     = joblib.load('../data/models/random_forest.pkl')
xgb    = joblib.load('../data/models/xgboost.pkl')
mlp    = joblib.load('../data/models/mlp.pkl')
scaler = joblib.load('../data/models/scaler_mlp.pkl')

train = pd.read_parquet('../data/clean/train.parquet')
test  = pd.read_parquet('../data/clean/test.parquet')

FEATURES = ['tipo_eve','tipo_veh','g_hora_5','dia_sem_ocu',
            'sexo_per','edad_quinquenales','mayor_menor','depto_ocu']
TARGET = 'fall_les'

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]
X_test_sc = scaler.transform(X_test)

print('✓ Modelos y datos cargados correctamente')

✓ Modelos y datos cargados correctamente


## 2. Cálculo de métricas (reemplaza la carga de resultados)

In [3]:
# -- Calcular métricas para los 3 modelos ----------------------------------------------
def get_proba_clase(modelo, X, pos_label=1):
    """Devuelve la probabilidad de la clase pos_label según el orden de clases del modelo."""
    probas = modelo.predict_proba(X)
    idx = list(modelo.classes_).index(pos_label)
    return probas[:, idx]

def calcular_metricas(modelo, X, y, pos_label=1):
    y_pred = modelo.predict(X)
    proba = get_proba_clase(modelo, X, pos_label)   # probabilidad de la clase pos_label
    
    # Convertir la clase positiva (pos_label) a 1 y la restante a 0
    y_bin = (y == pos_label).astype(int)
    y_pred_bin = (y_pred == pos_label).astype(int)
    
    acc = accuracy_score(y, y_pred)   # Accuracy se mantiene con las etiquetas originales
    f1  = f1_score(y, y_pred, average='weighted')
    auc = roc_auc_score(y_bin, proba)   # <--- AHORA CON y_bin
    
    prec = precision_score(y_bin, y_pred_bin)
    rec  = recall_score(y_bin, y_pred_bin)
    
    return {
        'modelo': modelo.__class__.__name__,
        'accuracy': acc,
        'f1_score': f1,
        'roc_auc': auc,
        'precision_fallecido': prec,
        'recall_fallecido': rec
    }

# Definir la clase positiva para cada modelo
# - Random Forest y MLP fueron entrenados con y_train (clase fallecido = 1)
# - XGBoost fue entrenado con y_train - 1 (clase fallecido = 0)
metricas_rf  = calcular_metricas(rf, X_test, y_test, pos_label=1)
metricas_xgb = calcular_metricas(xgb, X_test, y_test - 1, pos_label=0)
metricas_mlp = calcular_metricas(mlp, X_test_sc, y_test, pos_label=1)

df_res = pd.DataFrame([metricas_rf, metricas_xgb, metricas_mlp])

print(df_res[['modelo','accuracy','f1_score','roc_auc',
              'precision_fallecido','recall_fallecido']].to_string(index=False))

                modelo  accuracy  f1_score  roc_auc  precision_fallecido  recall_fallecido
RandomForestClassifier  0.694906  0.723336 0.733619             0.338195          0.624454
         XGBClassifier  0.671833  0.705295 0.739163             0.326416          0.675400
         MLPClassifier  0.652582  0.688878 0.716201             0.311442          0.676492


## 3. Comparación de métricas

In [4]:
# -- Visualización 1: Comparación de métricas principales ----------------------------------------------
metricas = ['accuracy', 'f1_score', 'roc_auc']
labels   = ['Accuracy', 'F1-Score', 'ROC-AUC']
colores  = ['#5B8DEF', '#F4A261', '#8338EC']

fig = go.Figure()
for i, row in df_res.iterrows():
    fig.add_trace(go.Bar(
        name=row['modelo'],
        x=labels,
        y=[row[m] for m in metricas],
        text=[f'{row[m]:.4f}' for m in metricas],
        textposition='auto',
        marker_color=colores[i]
    ))

fig.update_layout(
    title='Performance Metrics Comparison: All Models',
    barmode='group',
    yaxis=dict(range=[0.6, 0.80], title='Value'),
    xaxis_title='Metric',
    height=450,
    template='plotly_white',
    legend=dict(x=0.8, y=0.95)
)
fig.show()

In [5]:
# -- Visualización 2: Precision y Recall Fallecido ----------------------------------------------
fig2 = go.Figure()

for i, row in df_res.iterrows():
    fig2.add_trace(go.Bar(
        name=row['modelo'],
        x=['Precision (Fatality)', 'Recall (Fatality)'],
        y=[row['precision_fallecido'], row['recall_fallecido']],
        text=[f'{row["precision_fallecido"]:.2f}', f'{row["recall_fallecido"]:.2f}'],
        textposition='outside',
        marker_color=colores[i]
    ))

fig2.update_layout(
    title='Precision and Recall: Fatality Class',
    barmode='group',
    yaxis=dict(range=[0, 0.8], title='Value'),
    xaxis_title='Metric',
    height=450,
    template='plotly_white',
    legend=dict(x=0.8, y=0.95)
)
fig2.show()

In [6]:
# -- Curvas ROC y AUC (usando la clase positiva correcta para cada modelo) ----------
y_test_bin = (y_test == 1).astype(int)

def get_proba_clase(modelo, X, pos_label=1):
    probas = modelo.predict_proba(X)
    idx = list(modelo.classes_).index(pos_label)
    return probas[:, idx]

# Random Forest (pos_label=1)
prob_rf = get_proba_clase(rf, X_test, pos_label=1)
fpr_rf, tpr_rf, _ = roc_curve(y_test_bin, prob_rf)
auc_rf = roc_auc_score(y_test_bin, prob_rf)

# XGBoost (pos_label=0)
prob_xgb = get_proba_clase(xgb, X_test, pos_label=0)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test_bin, prob_xgb)
auc_xgb = roc_auc_score(y_test_bin, prob_xgb)

# MLP (pos_label=1)
prob_mlp = get_proba_clase(mlp, X_test_sc, pos_label=1)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test_bin, prob_mlp)
auc_mlp = roc_auc_score(y_test_bin, prob_mlp)

print(f'ROC-AUC Random Forest : {auc_rf:.4f}')
print(f'ROC-AUC XGBoost       : {auc_xgb:.4f}')
print(f'ROC-AUC MLP           : {auc_mlp:.4f}')

# Verificación del orden de clases
print("Clases RF :", rf.classes_)
print("Clases XGB:", xgb.classes_)
print("Clases MLP:", mlp.classes_)

ROC-AUC Random Forest : 0.7336
ROC-AUC XGBoost       : 0.7392
ROC-AUC MLP           : 0.7162
Clases RF : [1 2]
Clases XGB: [0 1]
Clases MLP: [1 2]


In [7]:
# -- Visualización 3: Curvas ROC superpuestas ----------------------------------------------
fig_roc = go.Figure()

fig_roc.add_trace(go.Scatter(
    x=fpr_rf, y=tpr_rf, mode='lines',
    name=f'Random Forest (AUC = {auc_rf:.4f})',
    line=dict(color='#5B8DEF', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=fpr_xgb, y=tpr_xgb, mode='lines',
    name=f'XGBoost (AUC = {auc_xgb:.4f})',
    line=dict(color='#F4A261', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=fpr_mlp, y=tpr_mlp, mode='lines',
    name=f'MLP (AUC = {auc_mlp:.4f})',
    line=dict(color='#8338EC', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    name='Baseline (AUC = 0.5)',
    line=dict(color='gray', width=1.5, dash='dash')
))

fig_roc.update_layout(
    title='ROC Curves: Model Comparison',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=500,
    template='plotly_white',
    legend=dict(x=0.55, y=0.1)
)
fig_roc.show()

In [8]:
# -- Visualización 4: Tabla resumen ----------------------------------------------
df_resumen = df_res[['modelo','accuracy','f1_score','roc_auc',
                     'precision_fallecido','recall_fallecido']].copy()

fig_tabla = go.Figure(go.Table(
    columnwidth=[160, 110, 110, 110, 170, 160],
    header=dict(
        values=['Model','Accuracy','F1-Score','ROC-AUC',
                'Precision (Fatality)','Recall (Fatality)'],
        fill_color='#2c2c2a',
        font=dict(color='white', size=11),
        align='center',
        height=35
    ),
    cells=dict(
        values=[
            df_resumen['modelo'],
            df_resumen['accuracy'].apply(lambda x: f'{x:.4f}'),
            df_resumen['f1_score'].apply(lambda x: f'{x:.4f}'),
            df_resumen['roc_auc'].apply(lambda x: f'{x:.4f}'),
            df_resumen['precision_fallecido'].apply(lambda x: f'{x:.4f}'),
            df_resumen['recall_fallecido'].apply(lambda x: f'{x:.4f}')
        ],
        fill_color=[['#E6F1FB','#FAEEDA','#E1F5EE']],
        font=dict(size=11),
        align='center',
        height=32
    )
))
fig_tabla.update_layout(
    title='Comparative Summary: All Models',
    height=220,
    margin=dict(l=0, r=0, t=40, b=0),
    template='plotly_white'
)
fig_tabla.show()

## 4. Diagnóstico de overfitting

In [9]:
# -- Diagnóstico de overfitting los 3 modelos ----------------------------------------------
X_train_sc = scaler.transform(X_train)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

modelos_dict = {
    'Random Forest': (rf, X_train, X_test, y_train, y_test, None),
    'XGBoost'      : (xgb, X_train, X_test, y_train - 1, y_test - 1, None),
    'MLP'          : (mlp, X_train_sc, X_test_sc, y_train, y_test, sample_weights),
}

print(f'{"Modelo":<15} {"Train":>8} {"Test":>8} {"Diff":>8} {"Estado"}')
print('-' * 55)
for nombre, (modelo, Xtr, Xte, ytr, yte, sw) in modelos_dict.items():
    acc_tr = accuracy_score(ytr, modelo.predict(Xtr))
    acc_te = accuracy_score(yte, modelo.predict(Xte))
    diff   = abs(acc_tr - acc_te)
    estado = '✓ OK' if diff < 0.05 else '⚠ Revisar'
    print(f'{nombre:<15} {acc_tr:>8.4f} {acc_te:>8.4f} {diff:>8.4f} {estado}')

Modelo             Train     Test     Diff Estado
-------------------------------------------------------


Random Forest     0.7230   0.6949   0.0281 ✓ OK
XGBoost           0.6879   0.6718   0.0161 ✓ OK


MLP               0.6511   0.6526   0.0015 ✓ OK


### Diagnóstico de overfitting

| Modelo | Train | Test | Diferencia | Estado |
|---|---|---|---|---|
| Random Forest | 0.7230 | 0.6949 | 0.0281 | ✓ OK |
| XGBoost | 0.6879 | 0.6718 | 0.0161 | ✓ OK |
| MLP | 0.6511 | 0.6526 | 0.0015 | ✓ OK |

Los 3 modelos generalizan correctamente : diferencia menor a 0.05.
Sin SMOTE, la brecha train/test se redujo significativamente.

## 5. Selección del modelo final

**Modelo seleccionado: Random Forest**

**Criterios de evaluación:**
1. ROC-AUC : poder discriminativo general
2. F1-Score : balance entre precision y recall
3. Overfitting : generalización del modelo
4. Importancia : distribución balanceada de features
5. Interpretabilidad : explicabilidad de predicciones

| Modelo | ROC-AUC | F1-Score | Overfitting | Importancia | Interpretabilidad |
|---|---|---|---|---|---|
| Random Forest | 0.7336 | 0.7233 | 0.028 ✓ | Balanceada | Alta |
| XGBoost | 0.7392 | 0.7053 | 0.016 ✓ | Predominio de la variable Sexo | Media |
| MLP | 0.7162 | 0.6889 | 0.002 ✓ | No interpretable | Baja |

Random Forest fue seleccionado como modelo final debido a su mejor equilibrio entre Accuracy, F1-Score, capacidad discriminativa, estabilidad de generalización e interpretabilidad, pese a que XGBoost obtuvo un ROC-AUC ligeramente superior.